# **Optical flow method to detect motion in video**

---



In [ ]:
!pip install opencv-python-headless numpy

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving WhatsApp Video 2026-03-22 at 8.55.26 PM (1).mp4 to WhatsApp Video 2026-03-22 at 8.55.26 PM (1).mp4


In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

def calculate_speed(flow, scale_factor, fps):
    magnitudes = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)
    avg_magnitude = np.mean(magnitudes)
    speed_m_per_s = avg_magnitude * scale_factor * fps
    speed_km_per_h = speed_m_per_s * 3.6
    return speed_km_per_h

def detect_vehicles(frame, fg_mask):
    contours, _ = cv2.findContours(fg_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    vehicle_contours = []

    for contour in contours:
        if cv2.contourArea(contour) > 1000:
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / float(h)
            if 1.2 > aspect_ratio > 0.3:
                vehicle_contours.append((x, y, w, h))

    return vehicle_contours

def draw_bounding_box(frame, vehicles, flow, scale_factor, fps):
    for (x, y, w, h) in vehicles:
        roi_flow = flow[y:y+h, x:x+w]
        speed = calculate_speed(roi_flow, scale_factor, fps)

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, f"{speed:.2f} km/h", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

def main(video_path, scale_factor, fps):
    cap = cv2.VideoCapture(video_path)

    ret, prev_frame = cap.read()
    if not ret:
        print("Error: Cannot read video")
        return

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    background_subtractor = cv2.createBackgroundSubtractorMOG2()

    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Optical Flow
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, gray, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        # Background subtraction
        fg_mask = background_subtractor.apply(gray)

        vehicles = detect_vehicles(frame, fg_mask)
        draw_bounding_box(frame, vehicles, flow, scale_factor, fps)

        # Show every 5th frame (important for Colab speed)
        if frame_count % 5 == 0:
            cv2_imshow(frame)

        prev_gray = gray.copy()
        frame_count += 1

    cap.release()

video_path = "/content/WhatsApp Video 2026-03-22 at 8.55.26 PM (1).mp4"   # Uploaded file name
scale_factor = 0.05         # Adjust based on camera calibration
fps = 30

main(video_path, scale_factor, fps)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/murugang2527/computer-vision.git

Cloning into 'computer-vision'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 67 (delta 24), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (67/67), 8.61 MiB | 14.48 MiB/s, done.
Resolving deltas: 100% (24/24), done.


In [ ]:
!cp "/content/Optical_flow_method.ipynb" "/content/computer-vision/"

cp: cannot stat '/content/Optical_flow_method.ipynb': No such file or directory
